# C3 — Versionner avec Git

**Objectif** : comprendre les mécanismes de Git en les simulant en Python pur.

Pas besoin d'installer Git : tout tourne en mémoire, compatible Google Colab.

## Étape 1 — Représenter un dépôt Git

In [ ]:
from datetime import datetime

class Commit:
    def __init__(self, message, fichiers, parent=None):
        self.message = message
        self.fichiers = dict(fichiers)  # snapshot
        self.parent = parent
        self.sha = hex(hash((message, datetime.now().isoformat())))[-8:]

class Depot:
    def __init__(self):
        self.commits = []          # historique
        self.staging = {}          # index (git add)
        self.working = {}          # fichiers de travail

    def modifier_fichier(self, nom, contenu):
        self.working[nom] = contenu

    def add(self, nom):
        if nom in self.working:
            self.staging[nom] = self.working[nom]

    def commit(self, message):
        if not self.staging:
            print('Rien à committer.')
            return
        parent = self.commits[-1] if self.commits else None
        c = Commit(message, self.staging, parent)
        self.commits.append(c)
        self.staging = {}
        print(f'[{c.sha}] {message}')

# Test
depot = Depot()
depot.modifier_fichier('ex01.py', 'def bonjour(): print("hello")')
depot.add('ex01.py')
depot.commit('ex01 : fonction bonjour créée')

## Étape 2 — git log

In [ ]:
def git_log(depot):
    if not depot.commits:
        print('Aucun commit.')
        return
    for c in reversed(depot.commits):
        print(f'{c.sha}  {c.message}')

# Ajouter d'autres commits pour avoir un historique
depot.modifier_fichier('ex01.py', 'def bonjour(): print("Hello World!")')
depot.add('ex01.py')
depot.commit('ex01 : message amélioré')

depot.modifier_fichier('ex02.py', 'def somme(a, b): return a + b')
depot.add('ex02.py')
depot.commit('ex02 : fonction somme ajoutée')

print('--- git log ---')
git_log(depot)

## Étape 3 — git diff (comparaison deux commits)

In [ ]:
def git_diff(commit_a, commit_b):
    """Compare les fichiers entre deux commits."""/
    tous = set(commit_a.fichiers) | set(commit_b.fichiers)
    for f in sorted(tous):
        avant = commit_a.fichiers.get(f, '(absent)')
        apres = commit_b.fichiers.get(f, '(absent)')
        if avant != apres:
            print(f'--- {f}')
            print(f'- {avant}')
            print(f'+ {apres}')
        else:
            print(f'  {f} (inchangé)')

print('--- git diff commit[0] commit[1] ---')
git_diff(depot.commits[0], depot.commits[1])

## Étape 4 — Branches

In [ ]:
class DepotAvecBranches(Depot):
    def __init__(self):
        super().__init__()
        self.branches = {'main': []}
        self.branche_courante = 'main'

    def checkout_b(self, nom):
        self.branches[nom] = list(self.commits)  # fork de l'historique
        self.branche_courante = nom
        print(f'Basculé sur la branche {nom}')

    def commit(self, message):
        if not self.staging:
            print('Rien à committer.')
            return
        parent = self.commits[-1] if self.commits else None
        c = Commit(message, self.staging, parent)
        self.commits.append(c)
        self.branches[self.branche_courante].append(c)
        self.staging = {}
        print(f'[{self.branche_courante}] [{c.sha}] {message}')

d2 = DepotAvecBranches()
d2.modifier_fichier('main.py', 'v1')
d2.add('main.py')
d2.commit('init')

d2.checkout_b('marie-exercices')
d2.modifier_fichier('ex01.py', 'solution marie')
d2.add('ex01.py')
d2.commit('ex01 : solution marie')

print('\nBranches :', list(d2.branches.keys()))

## Étape 5 — git status

In [ ]:
def git_status(depot):
    # Fichiers staged (git add fait)
    if depot.staging:
        print('Changements prêts à être commités (staged) :')
        for f in depot.staging:
            print(f'  modifié : {f}')
    else:
        print('Rien en staging.')

    # Fichiers modifiés non stagés
    dernier_snapshot = depot.commits[-1].fichiers if depot.commits else {}
    non_stagues = [
        f for f, v in depot.working.items()
        if v != dernier_snapshot.get(f) and f not in depot.staging
    ]
    if non_stagues:
        print('\nChangements non stagés :')
        for f in non_stagues:
            print(f'  modifié : {f}')

depot.modifier_fichier('ex03.py', 'nouveau fichier non ajouté')
depot.modifier_fichier('ex01.py', 'modification non stagée')
git_status(depot)